<a href="https://colab.research.google.com/github/Phils-dev/RAG_Homework/blob/main/PX4_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 드론 운용기록 분석 및 비행안전 기술문서 RAG 지원 시스템

- 과목 : 생성형 AI 활용
- 작성자 :
- 개발환경 : Google Colab, Python
- 대상기체 : Holybro X500 V2 + Pixhawk 6C
- 비행제어 소프트웨어 : PX4 Autopilot
- 기술문서 버전 : PX4 v1.17.0

## 개발 목적

PX4 공식 기술문서를 활용한 RAG 시스템을 구축하고,
드론 운용일지와 점검이력을 분석하여 정비 점검항목 및
비행안전 관련 기술정보를 제공한다.

※ 본 프로그램의 운용일지 및 정비주기 지침은
교육용 가상 데이터이며, 실제 비행 적합성 판단에 사용하지 않는다.

# PART 1. 개발 환경 구성

## 01. 개발 환경 설정

RAG 시스템 구축 및 드론 운용기록 분석에 필요한
Python 패키지를 설치한다.

In [1]:

# 패키지 설치
%pip install -q \
    "langchain>=1.0,<2.0" \
    "langchain-core>=1.4.7,<2.0" \
    "langchain-text-splitters>=1.1.2,<2.0" \
    langchain-openai \
    langchain-chroma \
    chromadb \
    langchain-huggingface \
    sentence-transformers \
    requests==2.32.4 \
    opentelemetry-api==1.42.1 \
    opentelemetry-sdk==1.42.1 \
    opentelemetry-exporter-otlp-proto-grpc==1.42.1 \
    "oauthlib>=3.2.0,<4" \
    jedi


In [2]:
# 패키지 의존성 검사
%pip check

No broken requirements found.


In [3]:
# 설치 환경 확인
import sys
from importlib.metadata import version

# Python 버전 출력
print("Python :", sys.version.split()[0])

# 패키지 목록 정의
packages = [
    "langchain",
    "langchain-core",
    "langchain-text-splitters",
    "langchain-openai",
    "langchain-chroma",
    "chromadb",
    "langchain-huggingface",
    "sentence-transformers",
    "pandas",
    "requests",
    "opentelemetry-api",
    "opentelemetry-sdk"
]

# 패키지 버전 출력
for package in packages:
    print(f"{package} : {version(package)}")

Python : 3.13.18
langchain : 1.4.8
langchain-core : 1.8.5
langchain-text-splitters : 1.2.0
langchain-openai : 1.8.5
langchain-chroma : 1.2.0
chromadb : 1.5.9
langchain-huggingface : 1.3.0
sentence-transformers : 5.7.0
pandas : 2.2.3
requests : 2.32.4
opentelemetry-api : 1.42.1
opentelemetry-sdk : 1.42.1



-----------------------------------------------------------



## 02. 라이브러리 불러오기

RAG 시스템 구축, 기술문서 처리, 운용일지 분석 및
점검 보고서 생성에 필요한 라이브러리를 불러온다.


In [4]:

# 기본 라이브러리
import os
import re
import json
import time
import hashlib
from pathlib import Path

# 데이터 분석 라이브러리
import pandas as pd

# Colab 라이브러리
from google.colab import userdata, files
from IPython.display import display, Markdown

# 문서 처리 라이브러리
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 임베딩 라이브러리
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

# 벡터 DB 라이브러리
from langchain_chroma import Chroma

# LLM 및 RAG 라이브러리
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 라이브러리 로드 결과 출력
print("라이브러리 로드 완료")


라이브러리 로드 완료
